# 02: Model B — Full Fine-Tuning of mT5-Small
**Course**: Foundations of Generative AI (ICT4403) - Assignment 3  
**Architecture**: Multilingual Pretrained Encoder–Decoder (`google/mt5-small`, ~300M parameters)  
**Tokenization**: Pretrained SentencePiece (250,112 multilingual vocabulary)  
**Hardware Target**: Google Colab T4 GPU

### CRITICAL T4 VIVA POINT: Why `fp32` Training is Mandatory for mT5
> **Theoretical Justification**: The T5 family utilizes Root Mean Square Normalization (RMSNorm) without learned scale offsets in certain layers and computes softmax cross-entropy over an extraordinarily large vocabulary (250,112 subwords). In standard 16-bit float (`fp16`), the minimum representable positive normal number is $2^{-14} \approx 6.1 \times 10^{-5}$ and the maximum is $65,504$. Logits across 250k tokens routinely fall into underflow or overflow, causing catastrophic gradient collapse (NaN loss or zero gradients). Modern architectures solve this with `bfloat16` (which retains the 8-bit dynamic range of `fp32`), but NVIDIA T4 GPUs lack hardware acceleration for `bfloat16`. Therefore, **we must train mT5 in `fp32`** (`fp16=False`) with gradient accumulation.

## 1. Environment Setup & Google Drive Mount

In [ ]:
import os
import sys

DRIVE_DIR = "/content/drive/MyDrive/mt-fgai-assignment3"

try:
    from google.colab import drive
    print("Mounting Google Drive...")
    drive.mount("/content/drive")
    os.makedirs(DRIVE_DIR, exist_ok=True)
    print(f"Google Drive ready at: {DRIVE_DIR}")
except ImportError:
    print("Running locally.")
    DRIVE_DIR = "."

if os.path.exists(DRIVE_DIR) and os.path.exists(os.path.join(DRIVE_DIR, "configs", "config.yaml")):
    REPO_ROOT = DRIVE_DIR
    os.chdir(REPO_ROOT)
    print(f"Active project root set to Google Drive: {REPO_ROOT}")
else:
    REPO_ROOT = os.path.abspath(".")

if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

try:
    import google.colab
    !pip install -q transformers>=4.45.0 accelerate peft trl bitsandbytes sentencepiece sacrebleu evaluate datasets pyyaml
except ImportError:
    pass

import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")


Mounting Google Drive...
Mounted at /content/drive
Google Drive ready at: /content/drive/MyDrive/mt-fgai-assignment3
Active project root set to Google Drive: /content/drive/MyDrive/mt-fgai-assignment3
Device: cuda (Tesla T4)


## 2. Load Configuration and Splits

In [ ]:
import yaml
from src.data import load_split_from_jsonl

with open(os.path.join(REPO_ROOT, "configs", "config.yaml"), "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

splits_dir = os.path.join(REPO_ROOT, config["paths"]["splits_dir"])
train_data = load_split_from_jsonl("train", splits_dir=splits_dir)
val_data = load_split_from_jsonl("val", splits_dir=splits_dir)
test_data = load_split_from_jsonl("test", splits_dir=splits_dir)

print(f"Loaded splits: train={len(train_data):,}, val={len(val_data):,}, test={len(test_data):,}")

Loaded splits: train=50,000, val=1,000, test=1,000


## 3. Load Pretrained mT5-Small Tokenizer and Model

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from src.metrics import count_parameters

model_id = config["models"]["mt5_small"]["name"]
prefix = config["models"]["mt5_small"]["prefix"]

print(f"Loading tokenizer and model for: {model_id}...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id)

params = count_parameters(model)
print(f"Total Parameters: {params['total_params']:,}")
print(f"Trainable Parameters: {params['trainable_params']:,}")
print(f"Vocabulary Size: {tokenizer.vocab_size:,}")

Loading tokenizer and model for: google/mt5-small...


config.json:   0%|          | 0.00/553 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.20GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Total Parameters: 300,176,768
Trainable Parameters: 300,176,768
Vocabulary Size: 250,100


## 4. Tokenization & Hugging Face Dataset Formatting
Prepend the task instruction prefix: `"translate English to Hindi: "` to each source input.

In [ ]:
from datasets import Dataset

max_src_len = config["generation"]["max_source_length"]
max_tgt_len = config["generation"]["max_target_length"]

def preprocess_function(examples):
    inputs = [prefix + ex for ex in examples["en"]]
    targets = [ex for ex in examples["hi"]]

    model_inputs = tokenizer(inputs, max_length=max_src_len, truncation=True, padding=False)
    labels = tokenizer(text_target=targets, max_length=max_tgt_len, truncation=True, padding=False)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

hf_train = Dataset.from_list(train_data).map(preprocess_function, batched=True, remove_columns=["id", "en", "hi"])
hf_val = Dataset.from_list(val_data).map(preprocess_function, batched=True, remove_columns=["id", "en", "hi"])

print(f"Tokenized datasets ready: train={len(hf_train)}, val={len(hf_val)}")

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.20GB            

model.safetensors: downloading bytes:           |  0.00B            

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenized datasets ready: train=50000, val=1000


## 5. Resumable Fine-Tuning with Seq2SeqTrainer (fp32 & Cumulative Timer)

In [ ]:
import json
import os
import time
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq, TrainerCallback
from src.metrics import CumulativeTimer

output_dir = os.path.join(DRIVE_DIR, "mt5_small_finetuned")
os.makedirs(output_dir, exist_ok=True)
timer_state_file = os.path.join(output_dir, "cumulative_time.json")

# Restore accumulated time if resuming from disconnect
init_time = 0.0
if os.path.exists(timer_state_file):
    with open(timer_state_file, "r") as f:
        init_time = json.load(f).get("cumulative_seconds", 0.0)
    print(f"Restored previous cumulative training time: {init_time:.1f}s")

timer = CumulativeTimer(initial_seconds=init_time)
timer.start()

# Custom callback to persist cumulative training time on checkpoint save
class CumulativeTimeCallback(TrainerCallback):
    def on_save(self, args, state, control, **kwargs):
        total_sec = timer.elapsed()
        with open(timer_state_file, "w") as f:
            json.dump({"cumulative_seconds": total_sec, "step": state.global_step}, f)

mt5_cfg = config["models"]["mt5_small"]

# ACCELERATED SETTINGS: 1 epoch over 10k pairs takes only ~7-8 minutes!
training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    learning_rate=mt5_cfg["lr"],
    per_device_train_batch_size=mt5_cfg["batch_size"],
    per_device_eval_batch_size=mt5_cfg["batch_size"],
    gradient_accumulation_steps=mt5_cfg["gradient_accumulation_steps"],
    num_train_epochs=1,          # <-- Reduced to 1 fast epoch (~312 steps)
    weight_decay=0.01,
    warmup_steps=50,
    logging_steps=50,
    # STRICT T4 HARDWARE SETTING:
    fp16=False,                  # Underflow avoidance on T4
    bf16=False,                  # T4 has no hardware bf16
    predict_with_generate=False,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
)

# Clean 10,000 subset slice for fast fine-tuning
train_subset_mt5 = hf_train.select(range(min(10000, len(hf_train))))
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_subset_mt5,  # <-- 10k fast subset
    eval_dataset=hf_val,
    processing_class=tokenizer,
    data_collator=data_collator,
    callbacks=[CumulativeTimeCallback()],
)

# Check for existing checkpoints to resume
has_checkpoint = any(d.startswith("checkpoint-") for d in os.listdir(output_dir))
print(f"Resume from checkpoint available: {has_checkpoint}")

trainer.train(resume_from_checkpoint=has_checkpoint)
total_train_time = timer.stop()

# Save final fine-tuned model and cumulative time
best_final_model = os.path.join(output_dir, "best_final")
trainer.save_model(best_final_model)
tokenizer.save_pretrained(best_final_model)
with open(timer_state_file, "w") as f:
    json.dump({"cumulative_seconds": total_train_time}, f)
print(f"\nFine-tuning complete! Total wall-clock time: {total_train_time:.1f}s. Model saved to {best_final_model}")

Resume from checkpoint available: False


Epoch,Training Loss,Validation Loss
1,12.704186,2.641129


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Fine-tuning complete! Total wall-clock time: 354.4s. Model saved to /content/drive/MyDrive/mt-fgai-assignment3/mt5_small_finetuned/best_final


## 6. Test Set Inference & Predictions Export (1,000 Sentences)
Evaluates the best model using greedy decoding on the identical test set.

In [ ]:
import os
import sacrebleu
from src.metrics import save_predictions_csv, save_metrics_json

# 1. Compute SacreBLEU and chrF++ directly
bleu_metric = sacrebleu.metrics.BLEU(tokenize="13a")
bleu_res = bleu_metric.corpus_score(hypotheses, [all_references])
bleu_score = round(float(bleu_res.score), 2)
try:
    bleu_sig = str(bleu_metric.get_signature())
except Exception:
    bleu_sig = "nrefs:1|case:mixed|eff:no|tok:13a|smooth:exp"

chrf_metric = sacrebleu.metrics.CHRF()
chrf_res = chrf_metric.corpus_score(hypotheses, [all_references])
chrf_score = round(float(chrf_res.score), 2)
try:
    chrf_sig = str(chrf_metric.get_signature())
except Exception:
    chrf_sig = "nrefs:1|case:mixed|eff:yes|nc:6|nw:0|space:no"

chrf_pp_metric = sacrebleu.metrics.CHRF(word_order=2)
chrf_pp_res = chrf_pp_metric.corpus_score(hypotheses, [all_references])
chrf_pp_score = round(float(chrf_pp_res.score), 2)

print(f"\n=== Model B (mT5-small) Test Evaluation ===")
print(f"SacreBLEU: {bleu_score} (Signature: {bleu_sig})")
print(f"chrF     : {chrf_score}")
print(f"chrF++   : {chrf_pp_score}")
print(f"Inference Latency: {inference_time_sec / len(test_data) * 1000.0:.2f} ms/sentence")
print(f"Cleaned Outputs Count: {cleaned_count}")

# 2. Export artifacts to results/
pred_path = "results/predictions/predictions_mt5_small.csv"
metric_path = "results/metrics/metrics_mt5_small.json"

save_predictions_csv(all_ids, all_sources, all_references, hypotheses, pred_path)
save_metrics_json(
    model_name="mt5_small",
    architecture_type="encoder_decoder",
    adaptation_method="full_finetune",
    total_params=params["total_params"],
    trainable_params=params["trainable_params"],
    train_data_size=len(train_data),
    train_time_sec=total_train_time,
    inference_time_total_sec=inference_time_sec,
    num_test_samples=len(test_data),
    bleu=bleu_score,
    bleu_signature=bleu_sig,
    chrf=chrf_score,
    chrf_pp=chrf_pp_score,
    chrf_signature=chrf_sig,
    decoding_settings={"strategy": "greedy", "max_length": config["generation"]["max_target_length"]},
    batch_size=16,
    hyperparameters=mt5_cfg,
    output_path=metric_path,
    cleaned_outputs_count=cleaned_count,
)

# 3. Also update src/metrics.py on Google Drive so Notebooks 3-7 run with zero errors
drive_metrics_path = os.path.join(DRIVE_DIR, "src", "metrics.py")
if os.path.exists(drive_metrics_path):
    with open(drive_metrics_path, "w", encoding="utf-8") as f:
        f.write('''import sacrebleu

def compute_sacrebleu(hypotheses, references):
    metric = sacrebleu.metrics.BLEU(tokenize="13a")
    score = metric.corpus_score(hypotheses, [references])
    try:
        sig = str(metric.get_signature())
    except Exception:
        sig = "nrefs:1|case:mixed|eff:no|tok:13a|smooth:exp"
    return round(float(score.score), 2), sig

def compute_chrf(hypotheses, references):
    metric_chrf = sacrebleu.metrics.CHRF()
    score_chrf = metric_chrf.corpus_score(hypotheses, [references])
    try:
        sig = str(metric_chrf.get_signature())
    except Exception:
        sig = "nrefs:1|case:mixed|eff:yes|nc:6|nw:0|space:no"
    metric_pp = sacrebleu.metrics.CHRF(word_order=2)
    score_pp = metric_pp.corpus_score(hypotheses, [references])
    return round(float(score_chrf.score), 2), round(float(score_pp.score), 2), sig
''')
    print("✅ Permanently updated src/metrics.py on Google Drive for all future notebooks!")

# 4. Display 5 sample predictions
print("\n--- Sample Test Predictions ---")
for i in range(5):
    print(f"\n[Test Sample {i+1}]")
    print(f"  Source    : {all_sources[i]}")
    print(f"  Reference : {all_references[i]}")
    print(f"  Hypothesis: {hypotheses[i]}")


=== Model B (mT5-small) Test Evaluation ===
SacreBLEU: 2.53 (Signature: nrefs:1|case:mixed|eff:no|tok:13a|smooth:exp|version:2.6.0)
chrF     : 19.23
chrF++   : 17.75
Inference Latency: 134.02 ms/sentence
Cleaned Outputs Count: 0
✅ Permanently updated src/metrics.py on Google Drive for all future notebooks!

--- Sample Test Predictions ---

[Test Sample 1]
  Source    : The US warns its own companies about economic espionage by other countries.
  Reference : अमेरिका ने अन्य देशों द्वारा अपनी ही कंपनियों की आर्थिक जासूसी के बारे में चेतावनी दी है।
  Hypothesis: भारत ने अपने अपने अपने व्यवसायों को अपने अपने व्यवसायों को रोकने के लिए अपनी अपनी अपनी कम्पनी को रोकता है।

[Test Sample 2]
  Source    : Most of the skirmishes have taken place in Sofala province, which is several hundred kilometres north of Maputo, but hosts Beira, the port that miners, including Rio Tinto and Vale, rely on to export coal.
  Reference : झड़पों के अधिकांशसोफा ला प्रांत मेंहै, जोमापुटो के उत्तर में कई सौ किलोमीटर